In [ ]:
# Install dependency
!pip install sqlalchemy

# ---------------- HR Management Demo ----------------
from sqlalchemy import create_engine, Column, Integer, String, DateTime, ForeignKey, Float
from sqlalchemy.orm import declarative_base, sessionmaker, relationship
from datetime import datetime

# Database setup
engine = create_engine("sqlite:///hris_full_demo.db", echo=False)
Base = declarative_base()
SessionLocal = sessionmaker(bind=engine)
db = SessionLocal()

# ---------------- Models ----------------
class Employee(Base):
    __tablename__ = "employees"
    id = Column(Integer, primary_key=True, index=True)
    name = Column(String, nullable=False)
    department = Column(String, nullable=False)
    email = Column(String, unique=True, nullable=False)

    attendance = relationship("Attendance", back_populates="employee")
    reviews = relationship("PerformanceReview", back_populates="employee")
    salaries = relationship("Salary", back_populates="employee")
    incentives = relationship("Incentive", back_populates="employee")

class Attendance(Base):
    __tablename__ = "attendance"
    id = Column(Integer, primary_key=True, index=True)
    employee_id = Column(Integer, ForeignKey("employees.id"))
    check_in = Column(DateTime, default=datetime.utcnow)
    check_out = Column(DateTime, nullable=True)
    employee = relationship("Employee", back_populates="attendance")

class PerformanceReview(Base):
    __tablename__ = "performance_reviews"
    id = Column(Integer, primary_key=True)
    employee_id = Column(Integer, ForeignKey("employees.id"))
    review_date = Column(DateTime, default=datetime.utcnow)
    rating = Column(Integer)   # 1-5 scale
    comments = Column(String)
    employee = relationship("Employee", back_populates="reviews")

class Salary(Base):
    __tablename__ = "salaries"
    id = Column(Integer, primary_key=True)
    employee_id = Column(Integer, ForeignKey("employees.id"))
    base_salary = Column(Float, nullable=False)
    allowances = Column(Float, default=0.0)
    effective_from = Column(DateTime, default=datetime.utcnow)
    employee = relationship("Employee", back_populates="salaries")

class Incentive(Base):
    __tablename__ = "incentives"
    id = Column(Integer, primary_key=True)
    employee_id = Column(Integer, ForeignKey("employees.id"))
    type = Column(String)   # e.g., "Bonus", "Award"
    amount = Column(Float, default=0.0)
    date_given = Column(DateTime, default=datetime.utcnow)
    employee = relationship("Employee", back_populates="incentives")

# Create tables
Base.metadata.create_all(bind=engine)

# ---------------- Functions ----------------
def add_employee(name, dept, email):
    emp = Employee(name=name, department=dept, email=email)
    db.add(emp)
    db.commit()
    print(f"✅ Added employee: {name}")

def list_employees():
    employees = db.query(Employee).all()
    print("\n👩‍💼 Employees:")
    for e in employees:
        print(f"ID: {e.id}, Name: {e.name}, Dept: {e.department}, Email: {e.email}")

def check_in(emp_id):
    att = Attendance(employee_id=emp_id, check_in=datetime.utcnow())
    db.add(att)
    db.commit()
    print(f"🕒 Employee {emp_id} checked in at {att.check_in}")

def check_out(att_id):
    att = db.query(Attendance).filter_by(id=att_id).first()
    if att and not att.check_out:
        att.check_out = datetime.utcnow()
        db.commit()
        print(f"🕒 Attendance {att_id} checked out at {att.check_out}")
    else:
        print("⚠️ Invalid attendance record")

def list_attendance():
    records = db.query(Attendance).all()
    print("\n📅 Attendance Records:")
    for r in records:
        print(f"ID:{r.id}, Emp:{r.employee_id}, In:{r.check_in}, Out:{r.check_out}")

def add_performance(emp_id, rating, comments):
    review = PerformanceReview(employee_id=emp_id, rating=rating, comments=comments)
    db.add(review)
    db.commit()
    print(f"⭐ Added performance review for Emp {emp_id} (Rating {rating})")

def list_performance(emp_id):
    reviews = db.query(PerformanceReview).filter_by(employee_id=emp_id).all()
    print(f"\n📊 Performance Reviews for Employee {emp_id}:")
    for r in reviews:
        print(f"Date:{r.review_date}, Rating:{r.rating}, Comments:{r.comments}")

def add_salary(emp_id, base, allowances=0):
    salary = Salary(employee_id=emp_id, base_salary=base, allowances=allowances)
    db.add(salary)
    db.commit()
    print(f"💰 Added salary record for Emp {emp_id}: {base}+{allowances}")

def list_salary(emp_id):
    salaries = db.query(Salary).filter_by(employee_id=emp_id).all()
    print(f"\n💵 Salary Records for Employee {emp_id}:")
    for s in salaries:
        print(f"Base:{s.base_salary}, Allowances:{s.allowances}, Effective:{s.effective_from}")

def add_incentive(emp_id, incentive_type, amount):
    inc = Incentive(employee_id=emp_id, type=incentive_type, amount=amount)
    db.add(inc)
    db.commit()
    print(f"🎁 Added incentive for Emp {emp_id}: {incentive_type} {amount}")

def list_incentives(emp_id):
    incentives = db.query(Incentive).filter_by(employee_id=emp_id).all()
    print(f"\n🏆 Incentives for Employee {emp_id}:")
    for i in incentives:
        print(f"Type:{i.type}, Amount:{i.amount}, Date:{i.date_given}")

# ---------------- Demo ----------------
add_employee("Aarav Sharma", "IT", "aarav@company.com")
add_employee("Meera Singh", "HR", "meera@company.com")

list_employees()

check_in(1)
check_in(2)
list_attendance()

check_out(1)
list_attendance()

add_performance(1, 5, "Excellent work this quarter")
add_performance(2, 4, "Good teamwork, needs improvement in deadlines")
list_performance(1)
list_performance(2)

add_salary(1, 50000, 5000)
add_salary(2, 45000, 4000)
list_salary(1)
list_salary(2)

add_incentive(1, "Bonus", 10000)
add_incentive(2, "Award", 5000)
list_incentives(1)
list_incentives(2)


✅ Added employee: Aarav Sharma
✅ Added employee: Meera Singh

👩‍💼 Employees:
ID: 1, Name: Aarav Sharma, Dept: IT, Email: aarav@company.com
ID: 2, Name: Meera Singh, Dept: HR, Email: meera@company.com
🕒 Employee 1 checked in at 2025-10-03 04:39:48.758339
🕒 Employee 2 checked in at 2025-10-03 04:39:48.769670

📅 Attendance Records:
ID:1, Emp:1, In:2025-10-03 04:39:48.758339, Out:None
ID:2, Emp:2, In:2025-10-03 04:39:48.769670, Out:None
🕒 Attendance 1 checked out at 2025-10-03 04:39:48.781410

📅 Attendance Records:
ID:1, Emp:1, In:2025-10-03 04:39:48.758339, Out:2025-10-03 04:39:48.781410
ID:2, Emp:2, In:2025-10-03 04:39:48.769670, Out:None
⭐ Added performance review for Emp 1 (Rating 5)
⭐ Added performance review for Emp 2 (Rating 4)

📊 Performance Reviews for Employee 1:
Date:2025-10-03 04:39:48.791699, Rating:5, Comments:Excellent work this quarter

📊 Performance Reviews for Employee 2:
Date:2025-10-03 04:39:48.799908, Rating:4, Comments:Good teamwork, needs improvement in deadlines
💰 A

/tmp/ipython-input-2008595871.py:80: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  att = Attendance(employee_id=emp_id, check_in=datetime.utcnow())
/tmp/ipython-input-2008595871.py:88: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  att.check_out = datetime.utcnow()


In [ ]:
# Install dependencies (only once per Colab session)
!pip install sqlalchemy gradio

import gradio as gr
from sqlalchemy import create_engine, Column, Integer, String, DateTime, ForeignKey, Float
from sqlalchemy.orm import declarative_base, sessionmaker, relationship
from datetime import datetime

# Database setup
engine = create_engine("sqlite:///hris_ui_demo.db", echo=False)
Base = declarative_base()
SessionLocal = sessionmaker(bind=engine)
db = SessionLocal()

# ---------------- Models ----------------
class Employee(Base):
    __tablename__ = "employees"
    id = Column(Integer, primary_key=True, index=True)
    name = Column(String, nullable=False)
    department = Column(String, nullable=False)
    email = Column(String, unique=True, nullable=False)
    attendance = relationship("Attendance", back_populates="employee")
    reviews = relationship("PerformanceReview", back_populates="employee")
    salaries = relationship("Salary", back_populates="employee")
    incentives = relationship("Incentive", back_populates="employee")

class Attendance(Base):
    __tablename__ = "attendance"
    id = Column(Integer, primary_key=True, index=True)
    employee_id = Column(Integer, ForeignKey("employees.id"))
    check_in = Column(DateTime, default=datetime.utcnow)
    check_out = Column(DateTime, nullable=True)
    employee = relationship("Employee", back_populates="attendance")

class PerformanceReview(Base):
    __tablename__ = "performance_reviews"
    id = Column(Integer, primary_key=True)
    employee_id = Column(Integer, ForeignKey("employees.id"))
    review_date = Column(DateTime, default=datetime.utcnow)
    rating = Column(Integer)
    comments = Column(String)
    employee = relationship("Employee", back_populates="reviews")

class Salary(Base):
    __tablename__ = "salaries"
    id = Column(Integer, primary_key=True)
    employee_id = Column(Integer, ForeignKey("employees.id"))
    base_salary = Column(Float, nullable=False)
    allowances = Column(Float, default=0.0)
    effective_from = Column(DateTime, default=datetime.utcnow)
    employee = relationship("Employee", back_populates="salaries")

class Incentive(Base):
    __tablename__ = "incentives"
    id = Column(Integer, primary_key=True)
    employee_id = Column(Integer, ForeignKey("employees.id"))
    type = Column(String)
    amount = Column(Float, default=0.0)
    date_given = Column(DateTime, default=datetime.utcnow)
    employee = relationship("Employee", back_populates="incentives")

Base.metadata.create_all(bind=engine)

# ---------------- Backend Functions ----------------
def add_employee(name, dept, email):
    emp = Employee(name=name, department=dept, email=email)
    db.add(emp)
    db.commit()
    return f"✅ Added employee: {name}"

def list_employees():
    employees = db.query(Employee).all()
    return "\n".join([f"{e.id}: {e.name} ({e.department}) - {e.email}" for e in employees]) or "No employees yet."

# ----- Attendance -----
def check_in(emp_id):
    att = Attendance(employee_id=int(emp_id), check_in=datetime.utcnow())
    db.add(att)
    db.commit()
    return f"🕒 Employee {emp_id} checked in."

def check_out(att_id):
    att = db.query(Attendance).filter_by(id=int(att_id)).first()
    if att and not att.check_out:
        att.check_out = datetime.utcnow()
        db.commit()
        return f"🕒 Attendance {att_id} checked out."
    return "⚠️ Invalid or already checked-out record."

def list_attendance_month(emp_id, month, year):
    records = db.query(Attendance).filter(
        Attendance.employee_id == int(emp_id),
        Attendance.check_in.between(f"{year}-{month:02d}-01", f"{year}-{month:02d}-31")
    ).all()
    return "\n".join([f"ID:{r.id}, In:{r.check_in}, Out:{r.check_out}" for r in records]) or "No records."

def edit_attendance(att_id, new_out):
    att = db.query(Attendance).filter_by(id=int(att_id)).first()
    if att:
        try:
            att.check_out = datetime.strptime(new_out, "%Y-%m-%d %H:%M:%S")
            db.commit()
            return f"✏️ Edited Attendance {att_id}"
        except:
            return "⚠️ Please use format YYYY-MM-DD HH:MM:SS"
    return "⚠️ Attendance not found."

# ----- Performance -----
def add_performance(emp_id, rating, comments):
    review = PerformanceReview(employee_id=int(emp_id), rating=int(rating), comments=comments)
    db.add(review)
    db.commit()
    return f"⭐ Added performance review for Emp {emp_id}."

def list_performance_month(emp_id, month, year):
    reviews = db.query(PerformanceReview).filter(
        PerformanceReview.employee_id == int(emp_id),
        PerformanceReview.review_date.between(f"{year}-{month:02d}-01", f"{year}-{month:02d}-31")
    ).all()
    return "\n".join([f"ID:{r.id}, {r.review_date.date()} - Rating:{r.rating}, {r.comments}" for r in reviews]) or "No reviews."

def edit_performance(review_id, new_rating, new_comments):
    rev = db.query(PerformanceReview).filter_by(id=int(review_id)).first()
    if rev:
        rev.rating = int(new_rating)
        rev.comments = new_comments
        db.commit()
        return f"✏️ Edited Performance Review {review_id}"
    return "⚠️ Review not found."

# ----- Salary -----
def add_salary(emp_id, base, allowances=0):
    salary = Salary(employee_id=int(emp_id), base_salary=float(base), allowances=float(allowances))
    db.add(salary)
    db.commit()
    return f"💰 Added salary for Emp {emp_id}."

def list_salary(emp_id):
    salaries = db.query(Salary).filter_by(employee_id=int(emp_id)).all()
    return "\n".join([f"{s.effective_from.date()} - Base:{s.base_salary}, Allowances:{s.allowances}" for s in salaries]) or "No salary records."

# ----- Incentives -----
def add_incentive(emp_id, incentive_type, amount):
    inc = Incentive(employee_id=int(emp_id), type=incentive_type, amount=float(amount))
    db.add(inc)
    db.commit()
    return f"🎁 Added incentive for Emp {emp_id}."

def list_incentives(emp_id):
    incentives = db.query(Incentive).filter_by(employee_id=int(emp_id)).all()
    return "\n".join([f"{i.date_given.date()} - {i.type}: {i.amount}" for i in incentives]) or "No incentives."


# ---------------- Gradio UI ----------------
with gr.Blocks() as demo:
    gr.Markdown("## 🏢 HR Management System (with Month-wise Editing)")

    with gr.Tab("Employees"):
        with gr.Row():
            name = gr.Textbox(label="Name")
            dept = gr.Textbox(label="Department")
            email = gr.Textbox(label="Email")
        add_btn = gr.Button("Add Employee")
        emp_list = gr.Textbox(label="Employees", interactive=False)
        add_btn.click(add_employee, [name, dept, email], emp_list)
        list_btn = gr.Button("Refresh List")
        list_btn.click(list_employees, outputs=emp_list)

    with gr.Tab("Attendance"):
        emp_id = gr.Textbox(label="Employee ID")
        check_in_btn = gr.Button("Check In")
        check_out_id = gr.Textbox(label="Attendance ID")
        check_out_btn = gr.Button("Check Out")

        # Month wise listing
        month = gr.Number(label="Month (1-12)")
        year = gr.Number(label="Year (YYYY)")
        att_list = gr.Textbox(label="Attendance Records", interactive=False)
        list_att_btn = gr.Button("Show Attendance for Month")
        list_att_btn.click(list_attendance_month, [emp_id, month, year], att_list)

        # Edit attendance
        att_edit_id = gr.Textbox(label="Attendance ID to Edit")
        new_out = gr.Textbox(label="New Check-out (YYYY-MM-DD HH:MM:SS)")
        edit_att_btn = gr.Button("Edit Attendance")
        edit_att_btn.click(edit_attendance, [att_edit_id, new_out], att_list)

        check_in_btn.click(check_in, [emp_id], att_list)
        check_out_btn.click(check_out, [check_out_id], att_list)

    with gr.Tab("Performance"):
        perf_emp = gr.Textbox(label="Employee ID")
        rating = gr.Number(label="Rating (1-5)")
        comments = gr.Textbox(label="Comments")
        perf_add = gr.Button("Add Review")
        perf_list = gr.Textbox(label="Performance Reviews", interactive=False)

        # Month wise listing
        perf_month = gr.Number(label="Month (1-12)")
        perf_year = gr.Number(label="Year (YYYY)")
        list_perf_btn = gr.Button("Show Reviews for Month")
        list_perf_btn.click(list_performance_month, [perf_emp, perf_month, perf_year], perf_list)

        # Edit performance
        review_edit_id = gr.Textbox(label="Review ID to Edit")
        new_rating = gr.Number(label="New Rating (1-5)")
        new_comments = gr.Textbox(label="New Comments")
        edit_perf_btn = gr.Button("Edit Review")
        edit_perf_btn.click(edit_performance, [review_edit_id, new_rating, new_comments], perf_list)

        perf_add.click(add_performance, [perf_emp, rating, comments], perf_list)

    with gr.Tab("Salary"):
        sal_emp = gr.Textbox(label="Employee ID")
        base = gr.Number(label="Base Salary")
        allow = gr.Number(label="Allowances")
        sal_add = gr.Button("Add Salary")
        sal_list = gr.Textbox(label="Salary Records", interactive=False)
        sal_add.click(add_salary, [sal_emp, base, allow], sal_list)
        sal_refresh = gr.Button("Refresh Salaries")
        sal_refresh.click(list_salary, [sal_emp], sal_list)

    with gr.Tab("Incentives"):
        inc_emp = gr.Textbox(label="Employee ID")
        inc_type = gr.Textbox(label="Type (Bonus/Award)")
        inc_amt = gr.Number(label="Amount")
        inc_add = gr.Button("Add Incentive")
        inc_list = gr.Textbox(label="Incentive Records", interactive=False)
        inc_add.click(add_incentive, [inc_emp, inc_type, inc_amt], inc_list)
        inc_refresh = gr.Button("Refresh Incentives")
        inc_refresh.click(list_incentives, [inc_emp], inc_list)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d58636751402efc524.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
